In [27]:
import os
import base64
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from PIL import Image
from io import BytesIO

# 1. Load Excel file
excel_path = ("C:/Users/omistaja/Desktop/AS Project/Data/"
    "Spreadsheets/street_signs_taipei_sheets.xlsx")

df = pd.read_excel(
    excel_path,
    sheet_name='OBSERVATION'
)

IMAGE_DIR = "../Data/Photos" 

# Group rows that share the exact same coordinates
grouped = df.groupby(['sign_latitude', 'sign_longitude'])

# Initialize your base map
m = folium.Map(location=[df['sign_latitude'].mean(), df['sign_longitude'].mean()], zoom_start=8)
marker_cluster = MarkerCluster().add_to(m)

# 2. Loop through each coordinate group
for (lat, lon), group_df in grouped:
    total_photos = len(group_df)
    images_html = ""
    
    # Process every photo belonging to this coordinate group
    for idx, row in group_df.iterrows():
        filename = str(row['sign_photo_filename']).strip()
        img_path = os.path.join(IMAGE_DIR, filename)
        
        if os.path.exists(img_path):
            try:
                # Open, resize, and compress the image so it doesn't freeze the notebook
                with Image.open(img_path) as img:
                    img.thumbnail((300, 300))  # Max bounds for popup display
                    
                    buffer = BytesIO()
                    img.save(buffer, format="JPEG", quality=70)  # Safe web compression
                    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
                
                # Append standard Base64 string directly into our group text block
                images_html += f"""
                <p style="margin: 5px 0 2px 0; font-size: 11px; color: #555;">📷 {filename}</p>
                <img src="data:image/jpeg;base64,{encoded}" width="240px" style="display: block; margin-bottom: 15px; border-radius: 4px;" alt="{filename}">
                """
            except Exception as e:
                images_html += f'<p style="color:orange; font-size:11px;">⚠️ Error loading {filename}: {str(e)}</p>'
        else:
            images_html += f'<p style="color:red; font-size:11px;">❌ File missing: "{filename}"</p>'
    
    # 3. Combine your image elements into a clean, scrollable window layout
    html_string = f"""
    <div style="font-family: sans-serif; width: 260px; max-height: 320px; overflow-y: auto; padding-right: 5px;">
        <h4 style="margin: 0 0 5px 0; color: #1a73e8;">📍 Gallery Location</h4>
        <span style="font-weight: bold; font-size: 12px; color: #333;">{total_photos} photos found here</span>
        <hr style="border: 0; border-top: 1px solid #ccc; margin: 5px 0 10px 0;">
        {images_html}
    </div>
    """
    
    # 4. Safely package and add to your map
    iframe = folium.IFrame(html=html_string, width=280, height=340)
    popup = folium.Popup(iframe, max_width=400)
    
    folium.Marker(location=[lat, lon], popup=popup).add_to(marker_cluster)

# Render map directly in your Jupyter cell
m

m.save('interactive_map.html')
